# Notebook 07 — Compute Extended Network Features

**Input:** `film_features_all.csv` + per-film `network_edges.csv`, `betweenness_null.csv`, `protagonist_metrics.csv`  
**Output:** `film_features_extended.csv` (adds burt_constraint, female_alter_betw_z, ego_density, alter_importance_ratio)  
**Script:** `scripts/compute_extended_features.py`  

Run this notebook after **notebook 06** (network analysis) whenever new films are added to the pipeline.

In [1]:
from pathlib import Path
import subprocess, sys

CLEAN = Path('..').resolve()
script = CLEAN / 'scripts' / 'compute_extended_features.py'
result = subprocess.run([sys.executable, str(script)], capture_output=True, text=True,
                        cwd=str(CLEAN))
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[:500])

beautyandthebeast_1991 (Belle): burt=0.320, female_alter_z=-0.07, ego_density=0.191, alter_importance=0.571
mulan_1998 (Mulan): burt=0.257, female_alter_z=0.80, ego_density=0.083, alter_importance=0.222
frozen_2013 (Anna): burt=0.333, female_alter_z=-0.21, ego_density=0.214, alter_importance=0.286
inside_out_2015 (Joy): burt=0.473, female_alter_z=-1.08, ego_density=0.191, alter_importance=0.143
zootopia_2016 (Hopps): burt=0.349, female_alter_z=-0.41, ego_density=0.046, alter_importance=0.250
encanto_2021 (Mirabel): burt=0.168, female_alter_z=-0.16, ego_density=0.023, alter_importance=0.083
raya_and_the_last_dragon_2021 (Raya): burt=0.436, female_alter_z=0.05, ego_density=0.400, alter_importance=0.400
elemental_2023 (Ember): burt=0.521, female_alter_z=-0.45, ego_density=0.107, alter_importance=0.125
toy_story_1995 (Woody): burt=0.493, female_alter_z=-0.49, ego_density=0.042, alter_importance=0.333
monsters_inc_2001 (Mike): burt=0.583, female_alter_z=-0.68, ego_density=N/A, alter_importa

## Verify output

In [2]:
import pandas as pd
ext = pd.read_csv(CLEAN / 'data' / '04_features' / 'film_features_extended.csv')
print(f'Shape: {ext.shape}')
print(f'Films: {ext.film_id.nunique()}')
print(f'Columns: {list(ext.columns)}')
ext[['film_id','protagonist','lead_gender','burt_constraint','female_alter_betw_z','ego_density']]

Shape: (19, 38)
Films: 18
Columns: ['film_id', 'year', 'protagonist', 'lead_gender', 'n_nodes', 'n_edges', 'density', 'reciprocity', 'avg_path_len', 'leading_eigenvalue', 'mean_clustering', 'homophily_obs', 'homophily_z', 'homophily_p', 'protag_samesex', 'protag_samesex_z', 'protag_samesex_p', 'protag_betweenness', 'protag_betw_z', 'protag_betw_p', 'keystone', 'keystone_gender', 'keystone_diff_gender', 'components_after_removal', 'n_isolated_after_removal', 'keystone_rung1', 'keystone_rung1_gender', 'keystone_rung2', 'keystone_rung2_gender', 'keystone_rung3', 'keystone_rung3_gender', 'keystone_changes_at_rung2', 'keystone_changes_at_rung3', 'burt_constraint', 'female_alter_betw_z', 'ego_density', 'alter_importance_ratio', 'n_samegender_alters']


,film_id,protagonist,lead_gender,burt_constraint,female_alter_betw_z,ego_density
0,beautyandthebeast_1991,Belle,F,0.3196,-0.069,0.1905
1,mulan_1998,Mulan,F,0.2570,0.797,0.0833
2,frozen_2013,Anna,F,0.3335,-0.214,0.2143
3,inside_out_2015,Joy,F,0.4732,-1.077,0.1905
4,zootopia_2016,Hopps,F,0.3486,-0.411,0.0458
5,encanto_2021,Mirabel,F,0.1679,-0.161,0.0227
6,raya_and_the_last_dragon_2021,Raya,F,0.4356,0.054,0.4000
7,elemental_2023,Ember,F,0.5208,-0.448,0.1071
8,toy_story_1995,Woody,M,0.4928,-0.488,0.0417
9,monsters_inc_2001,Mike,M,0.5830,-0.685,NaN


## What each metric means

| Metric | Definition | Range |
|---|---|---|
| `burt_constraint` | How embedded the protagonist is in a single dense clique (Burt 1992). Low = high brokerage across groups. | 0–1 |
| `female_alter_betw_z` | Mean betweenness z-score of the protagonist's female alters. Positive = female friends are structurally load-bearing. | unbounded |
| `ego_density` | Fraction of possible ties among the protagonist's alters that exist. 0 = star topology, 1 = tight clique. | 0–1 |
| `alter_importance_ratio` | Fraction of the protagonist's alters with above-average betweenness. | 0–1 |